# OHAYO - Serving Comparison

Fresh kernel, GPU T4, Internet. Attach the private comparison Input and existing NF4 checkpoint. Run cells 1-4. Both arms run sequentially in isolated processes on the same GPU. No Secrets, tunnel, training or backend changes. Export even if a run fails.


In [ ]:
from pathlib import Path
import hashlib, json, os, zipfile, subprocess, sys
from datetime import datetime, timezone
os.environ.update(TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
if globals().get('_ohayo_service'):
    raise RuntimeError('Restart Python first; do not run beside a loaded serving model.')
EXPECTED = '083d5536bc6e12983547c0402b7598f640bd673e0646f170e03c6f10d1ff7d5d'
PAYLOAD_HASHES = {'serving-input.zip': '4485b08d9aef43331a8b5aa60b0bc16966295cc95de3b086c01080a8384348d8', 'runner.py': 'c4c7ea56e5ce44f781770b4b1248879fc34e7f3f72f5c19e114b1b30f2239c59', 'cases.json': '0aea86ec4608130e70547ff9cddddf0130e33f6df5a184cd10f264babdb6613e'}
SERVING_MANIFEST_SHA256 = 'ecc0a5256e92a0585c0ab066ee6bb5dba9ffd5456be1f80f9f8903b56cadd621'
WORK = Path('/kaggle/working/ayo-paired-' + EXPECTED[:12])

def restore_expanded_serving(root):
    manifests = [p for p in root.rglob('manifest.json') if p.is_file() and not p.is_symlink()
                 and hashlib.sha256(p.read_bytes()).hexdigest()==SERVING_MANIFEST_SHA256]
    if not manifests:
        raise RuntimeError('Serving ZIP and matching expanded serving manifest are both missing from this Input.')
    source = manifests[0].parent.resolve()
    manifest_bytes = manifests[0].read_bytes()
    inventory = json.loads(manifest_bytes)['files']
    if len(inventory)>100 or sum(info['size_bytes'] for info in inventory.values())>160*1024**2:
        raise RuntimeError('Oversized expanded serving Input')
    contents = {'manifest.json':manifest_bytes}
    for name, info in inventory.items():
        relative = Path(name)
        if relative.is_absolute() or '..' in relative.parts or ':' in name or chr(92) in name:
            raise RuntimeError('Unsafe expanded serving path')
        path = source/relative
        if path.is_symlink() or not path.resolve().is_relative_to(source) or not path.is_file():
            raise RuntimeError('Missing or unsafe expanded serving file: '+name)
        data = path.read_bytes()
        if len(data)!=info['size_bytes'] or hashlib.sha256(data).hexdigest()!=info['sha256']:
            raise RuntimeError('Expanded serving checksum changed: '+name)
        contents[name] = data
    return contents

expanded_serving = None
inputs = [p for p in Path('/kaggle/input').rglob('comparison-input.zip') if hashlib.sha256(p.read_bytes()).hexdigest()==EXPECTED]
if inputs:
    with zipfile.ZipFile(inputs[0]) as archive:
        if set(archive.namelist()) != set(PAYLOAD_HASHES) or len(archive.namelist())!=3:
            raise RuntimeError('Wrong comparison package inventory')
        if sum(i.file_size for i in archive.infolist()) > 180*1024**2:
            raise RuntimeError('Oversized Input')
        payloads = {name:archive.read(name) for name in archive.namelist()}
else:
    roots = [p.parent for p in Path('/kaggle/input').rglob('runner.py')
             if hashlib.sha256(p.read_bytes()).hexdigest()==PAYLOAD_HASHES['runner.py']]
    if not roots:
        raise RuntimeError('Attach comparison-input.zip as a private dataset; keep the NF4 checkpoint Input.')
    payloads = {}
    for name in PAYLOAD_HASHES:
        path = roots[0]/name
        if path.is_symlink() or not path.resolve().is_relative_to(roots[0].resolve()):
            raise RuntimeError('Unsafe extracted Input')
        if name=='serving-input.zip' and not path.exists():
            expanded_serving = restore_expanded_serving(roots[0])
        elif path.is_file():
            payloads[name] = path.read_bytes()
        else:
            raise RuntimeError('Missing comparison Input file: '+name)
WORK.mkdir(exist_ok=True)
for name, data in payloads.items():
    if hashlib.sha256(data).hexdigest()!=PAYLOAD_HASHES[name]:
        raise RuntimeError('Comparison payload checksum changed')
    path = WORK/name
    if path.is_symlink() or (path.exists() and path.read_bytes()!=data):
        raise RuntimeError('Existing comparison package changed')
    path.write_bytes(data)
if expanded_serving is not None:
    expanded_root = WORK/'serving-expanded'
    for name,data in expanded_serving.items():
        path = expanded_root/name
        if path.is_symlink() or not path.resolve().is_relative_to(expanded_root.resolve()):
            raise RuntimeError('Unsafe serving staging destination')
        if path.exists() and path.read_bytes()!=data:
            raise RuntimeError('Existing expanded serving package changed')
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_bytes(data)
    print('SERVING_INPUT_VERIFIED: expanded manifest and every file checksum; original ZIP not required.')
del payloads
RUN = WORK / ('results-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RUN.mkdir()
print('INPUT_VERIFIED:', WORK)
print('Requests per arm:', len(json.loads((WORK/'cases.json').read_bytes())['requests']))

from pathlib import Path
import hashlib, json, os, zipfile

ARM = "tuned_rag"  # Rollback: restart Python, set "base_rag", then run cells 1-4.
assert ARM in ("base_rag", "tuned_rag")
if globals().get('_ohayo_service'):
    raise RuntimeError('Restart & Clear Cell Outputs before changing arm or reimporting this notebook.')
EXPECTED_PACKAGE_SHA256 = '4485b08d9aef43331a8b5aa60b0bc16966295cc95de3b086c01080a8384348d8'
EXPECTED_MANIFEST_SHA256 = 'ecc0a5256e92a0585c0ab066ee6bb5dba9ffd5456be1f80f9f8903b56cadd621'
EXPECTED_ADAPTER_SHA256 = '51713be1e630135099f95ed08c89b5be8669b75c9e1908a1346c5c9cc4942ae7'
candidates = [p for p in WORK.rglob('serving-input.zip')
              if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_PACKAGE_SHA256]
SERVING_PACKAGE_SHA256 = EXPECTED_PACKAGE_SHA256
SERVING_PACKAGE = Path('/kaggle/working/ayo-serving-' + EXPECTED_PACKAGE_SHA256[:12])
if candidates:
    with zipfile.ZipFile(candidates[0]) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)) or sum(i.file_size for i in archive.infolist()) > 160 * 1024**2:
            raise RuntimeError('Invalid or oversized serving ZIP')
        payloads = {name:archive.read(name) for name in names}
else:
    manifests = [p for p in WORK.rglob('manifest.json')
                 if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA256]
    if not manifests:
        raise RuntimeError('Add the matching private serving Input and keep the NF4 checkpoint Input too.')
    source_root = manifests[0].parent.resolve()
    inventory = json.loads(manifests[0].read_bytes())['files']
    if sum(info['size_bytes'] for info in inventory.values()) > 160 * 1024**2:
        raise RuntimeError('Oversized extracted Input')
    payloads = {'manifest.json':manifests[0].read_bytes()}
    for name in inventory:
        path = source_root / name
        if path.is_symlink() or not path.resolve().is_relative_to(source_root):
            raise RuntimeError('Unsafe extracted Input path')
        payloads[name] = path.read_bytes()
if hashlib.sha256(payloads['manifest.json']).hexdigest() != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('Serving manifest mismatch')
manifest = json.loads(payloads['manifest.json'])
if set(payloads) != set(manifest['files']) | {'manifest.json'}:
    raise RuntimeError('Unexpected package members')
for name, info in manifest['files'].items():
    relative = Path(name)
    if relative.is_absolute() or '..' in relative.parts or '\\' in name or ':' in name:
        raise RuntimeError('Unsafe package path')
    data = payloads[name]
    if len(data) != info['size_bytes'] or hashlib.sha256(data).hexdigest() != info['sha256']:
        raise RuntimeError('Package checksum mismatch')
    destination = SERVING_PACKAGE / relative
    if not destination.resolve().is_relative_to(SERVING_PACKAGE.resolve()) or destination.is_symlink():
        raise RuntimeError('Unsafe extraction destination')
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.read_bytes() != data:
        raise RuntimeError('Existing package changed; do not overwrite')
    destination.write_bytes(data)
del payloads
OHAYO_SERVING_ARM = ARM
os.environ.update(TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0')
print('INPUT_VERIFIED:', ARM, '| no training dataset, no base weights in this ZIP')


In [ ]:
import importlib.metadata
from packaging.specifiers import SpecifierSet
requirements = {'transformers':'==4.57.6','tokenizers':'==0.22.2','bitsandbytes':'==0.48.2',
                'accelerate':'==1.15.0','peft':'==0.17.1','safetensors':'==0.8.0','lm-format-enforcer':'==0.11.3'}
missing = []
for name, constraint in requirements.items():
    try:
        valid = importlib.metadata.version(name) in SpecifierSet(constraint)
    except importlib.metadata.PackageNotFoundError:
        valid = False
    if not valid:
        missing.append(name+constraint)
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
subprocess.run([sys.executable,'-c',
    'import torch,peft,bitsandbytes; from transformers import AutoTokenizer; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))'],check=True)
dependencies = {name:importlib.metadata.version(name) for name in ['torch',*requirements]}
(RUN/'dependencies.json').write_text(json.dumps(dependencies,indent=2))
print('ENVIRONMENT_READY; no model loaded in this kernel, no training.')


In [ ]:
import time
failures = []
for arm in ('base_rag','tuned_rag'):
    command = [sys.executable,'-u',str(WORK/'runner.py'),'--package',str(SERVING_PACKAGE),
               '--arm',arm,'--cases',str(WORK/'cases.json'),'--output',str(RUN/arm)]
    log = RUN/(arm+'.log')
    with log.open('w') as stream:
        process = subprocess.Popen(command,stdout=stream,stderr=subprocess.STDOUT)
        started = time.monotonic()
        while process.poll() is None:
            time.sleep(15)
            lines = log.read_text(errors='replace').splitlines()
            print(arm, round(time.monotonic()-started), 's |', lines[-1][-240:] if lines else 'STARTING',flush=True)
    if process.returncode:
        failures.append(arm)
        print('RUN_FAILED:',arm,'| evidence preserved; do not train again.')
if not failures:
    subprocess.run([sys.executable,str(WORK/'runner.py'),'--audit','--cases',str(WORK/'cases.json'),
                    '--output',str(RUN)],check=True)
else:
    (RUN/'failure.json').write_text(json.dumps({'failed_arms':failures}))
print('Run cell 4 to export all evidence, including failures. Backend configuration unchanged.')


In [ ]:
from IPython.display import FileLink, display
archive_path = WORK/(RUN.name+'.zip')
with zipfile.ZipFile(archive_path,'x',zipfile.ZIP_DEFLATED) as archive:
    archive.write(WORK/'cases.json','cases.json')
    for path in sorted(RUN.rglob('*')):
        if path.is_file() and 'source' not in path.relative_to(RUN).parts:
            archive.write(path,path.relative_to(RUN).as_posix())
receipt = dict(sha256=hashlib.sha256(archive_path.read_bytes()).hexdigest(),size_bytes=archive_path.stat().st_size,
               comparison_input_sha256=EXPECTED,serving_input_sha256=EXPECTED_PACKAGE_SHA256)
receipt_path = archive_path.with_suffix('.receipt.json')
receipt_path.write_text(json.dumps(receipt,indent=2))
display(FileLink(str(archive_path)),FileLink(str(receipt_path)))
print('Send ZIP + receipt. Semantic review and model selection follow; no training performed.')
